# v3c — COCO 2017 Fine-tuning (Best Model)

| Field | Value |
|---|---|
| Framework | PyTorch |
| Fine-tuning dataset | COCO 2017 — 118,287 train images, 5,000 val images, ~5 captions each |
| Starting weights | Phase 2d (v3b best checkpoint — Perceiver + GPT-2 + CrossAttention on Flickr30k) |
| Architecture | Unchanged from v3b — PercieverGptDecoder |
| Image Encoder | ConvNeXt-Tiny multi-scale → LMDB (COCO embeddings, same encoder as v3a/v3b) |
| Loss | CrossEntropyLoss(ignore_index=50256) |
| Inference | Top-k=20, temperature=0.7, retry logic (same as v3b) |
| BLEU-4 | **0.1341** (COCO val) |
| Status | Best model in the series — weights available on HuggingFace Hub |

## Overview

v3b demonstrated the right architecture — Perceiver + GPT-2 + CrossAttention at every layer — but the training data was the bottleneck. Flickr30k has 28,604 training images. The Perceiver and 12 CrossAttention blocks have millions of parameters that need diverse visual-language pairings to generalize.

v3c fine-tunes the Phase 2d checkpoint on **COCO 2017** in two stages:

**Phase 3a — COCO 50k subset:**
- 50,000 training images sampled from COCO train2017
- BLEU-4: **0.1331** on COCO val

**Phase 3a extended — Full COCO 118k:**
- All 118,287 COCO train2017 images (continuing from Phase 3a weights)
- BLEU-4: **0.1341** on COCO val — marginal gain; diminishing returns at this data scale

COCO 2017 offers:
- **118,287 training images** — 4× more than Flickr30k
- **~591K training captions** — 5 per image, longer and more detailed than Flickr30k captions
- **80 object categories** with consistent annotation — stronger visual-language grounding signal

No architecture changes are made in either stage. This is a pure fine-tuning run on a larger, higher-quality dataset.

**Final result: BLEU-4 0.1341** — a 2× improvement over v3b's Flickr30k BLEU (0.0656). Note that v3b was evaluated on Flickr30k val and v3c on COCO val — these are different distributions, so the jump reflects both the model improvement *and* the fact that the model is now evaluated on in-domain data.

The architecture is not re-documented here — see `v3b_perceiver_gpt2.ipynb` for the full Perceiver, CrossAttentionBlock, and GPT2withCrossAttention definitions.

## 1. Setup

In [ ]:
import os
import re
import json
import random
import numpy as np
import lmdb

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import GPT2Tokenizer, GPT2LMHeadModel

import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.corpus import words as nltk_words
nltk.download('punkt', quiet=True)
nltk.download('words', quiet=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token
PAD_ID = BOS_ID = EOS_ID = tokenizer.eos_token_id  # 50256

## 2. Dataset — COCO 2017

COCO annotations are stored in JSON files. The relevant structure:

```json
{
  "images":      [{"id": 1, "file_name": "000000000001.jpg"}, ...],
  "annotations": [{"image_id": 1, "caption": "A cat sitting on a mat."}, ...]
}
```

We build two dicts:
- `id_to_filename`: maps integer image ID → filename string (used as LMDB key)
- `coco_captions_dict`: maps filename → list of caption strings

COCO train2017 has 118,287 images with approximately 5 captions each. Val2017 has 5,000 images, also 5 captions each.

In [ ]:
COCO_ANN_TRAIN = '/kaggle/input/coco-2017-dataset/annotations/captions_train2017.json'
COCO_ANN_VAL   = '/kaggle/input/coco-2017-dataset/annotations/captions_val2017.json'

def load_coco_captions(ann_path):
    with open(ann_path, 'r') as f:
        data = json.load(f)

    id_to_filename = {img['id']: img['file_name'] for img in data['images']}
    captions_dict  = {}
    for ann in data['annotations']:
        fname = id_to_filename[ann['image_id']]
        captions_dict.setdefault(fname, []).append(ann['caption'])

    return captions_dict

coco_train_captions = load_coco_captions(COCO_ANN_TRAIN)
coco_val_captions   = load_coco_captions(COCO_ANN_VAL)

coco_train_keys = list(coco_train_captions.keys())
coco_val_keys   = list(coco_val_captions.keys())

print(f'COCO train images: {len(coco_train_keys)}')
print(f'COCO val images:   {len(coco_val_keys)}')
print(f'COCO train captions: {sum(len(v) for v in coco_train_captions.values()):,}')

sample_key = coco_train_keys[0]
print(f'\nSample key: {sample_key}')
print(f'Sample captions: {coco_train_captions[sample_key][:2]}')

## 3. COCO LMDB — Pre-computed ConvNeXt Embeddings

As in v3a/v3b, ConvNeXt-Tiny multi-scale features are pre-extracted and stored in LMDB. COCO's 118,287 training images require more shards than Flickr30k's 31,783.

Each key is an image filename (e.g., `000000000001.jpg`). Each value is a `float16` array of shape `(64, 768)` — identical format to the Flickr30k LMDB.

The extraction was run offline on Kaggle with a P100 GPU (same `ConvNextMultiScale` class from v3a, same transforms).

In [ ]:
COCO_LMDB_PARTS  = '/kaggle/input/datasets/huzefamerchant/coco-lmdb-part-{}'
COCO_N_SHARDS    = 12   # COCO is ~4× larger than Flickr30k

coco_envs = [
    lmdb.open(COCO_LMDB_PARTS.format(i), readonly=True, lock=False, readahead=False)
    for i in range(1, COCO_N_SHARDS + 1)
]

def get_coco_embedding(key):
    key_bytes = key.encode()
    for env in coco_envs:
        with env.begin() as txn:
            value = txn.get(key_bytes)
            if value is not None:
                return np.frombuffer(value, dtype=np.float16).reshape(64, 768)
    return None

# Sanity check
emb = get_coco_embedding(coco_train_keys[0])
print(f'Embedding shape: {emb.shape}')   # (64, 768)
print(f'Embedding dtype: {emb.dtype}')   # float16

## 4. Dataset and DataLoader

Same structure as v3b's `FlickrDataset`. The only difference is that COCO captions are stored as plain lists (not numbered dicts). `MAX_LEN=64` is sufficient — over 98% of COCO captions fit within 64 tokens.

In [ ]:
MAX_LEN = 64

class COCODataset(Dataset):
    def __init__(self, keys, captions_dict, get_emb_fn, max_len=MAX_LEN):
        self.samples    = [(k, cap) for k in keys for cap in captions_dict[k]]
        self.get_emb_fn = get_emb_fn
        self.max_len    = max_len

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        key, caption = self.samples[idx]
        image_tensor = torch.from_numpy(self.get_emb_fn(key).copy()).float()   # (64, 768)
        token_ids    = [BOS_ID] + tokenizer.encode(caption) + [EOS_ID]
        token_ids    = token_ids[:self.max_len]
        token_ids   += [PAD_ID] * (self.max_len - len(token_ids))
        return image_tensor, torch.tensor(token_ids, dtype=torch.long)


train_dataset    = COCODataset(coco_train_keys, coco_train_captions, get_coco_embedding)
val_dataset      = COCODataset(coco_val_keys,   coco_val_captions,   get_coco_embedding)

train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_dataloader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train samples: {len(train_dataset):,} | Val samples: {len(val_dataset):,}')

## 5. Model — Architecture (Unchanged from v3b)

The full architecture is defined in `v3b_perceiver_gpt2.ipynb`. Reproduced here for standalone executability.

In [ ]:
class Perceiver(nn.Module):
    def __init__(self, d_model=768, num_latents=16, num_layers=4, nhead=8):
        super().__init__()
        self.linear     = nn.Linear(768, 768)
        self.latents    = nn.Parameter(torch.randn(num_latents, d_model))
        self.cross_attn = nn.MultiheadAttention(d_model, nhead, batch_first=True)
        self.self_attn  = nn.ModuleList([
            nn.TransformerEncoderLayer(d_model, nhead, activation='gelu', batch_first=True)
            for _ in range(num_layers)
        ])

    def forward(self, x):                              # x: (B, 64, 768)
        x       = self.linear(x)
        B       = x.size(0)
        latents = self.latents.unsqueeze(0).repeat(B, 1, 1)
        latents, _ = self.cross_attn(query=latents, key=x, value=x)
        for layer in self.self_attn:
            latents = layer(latents)
        return latents                                 # (B, 16, 768)


class CrossAttentionBlock(nn.Module):
    def __init__(self, d_model=768, nhead=12):
        super().__init__()
        self.layernorm  = nn.LayerNorm(d_model)
        self.cross_attn = nn.MultiheadAttention(d_model, nhead, batch_first=True)

    def forward(self, perciever_image, x):
        x_normed   = self.layernorm(x)
        cross, _   = self.cross_attn(query=x_normed,
                                     key=perciever_image,
                                     value=perciever_image)
        return x + cross


class GPT2withCrossAttention(nn.Module):
    def __init__(self):
        super().__init__()
        gpt2             = GPT2LMHeadModel.from_pretrained('gpt2')
        self.gpt2_blocks = gpt2.transformer.h
        self.wte         = gpt2.transformer.wte
        self.wpe         = gpt2.transformer.wpe
        self.ln_f        = gpt2.transformer.ln_f
        self.lm_head     = gpt2.lm_head
        self.cross_attn  = nn.ModuleList([
            CrossAttentionBlock(d_model=768, nhead=12) for _ in range(12)
        ])

    def forward(self, perciever_image, caption_ids):
        B, T      = caption_ids.shape
        token_emb = self.wte(caption_ids)
        pos_ids   = torch.arange(T, device=caption_ids.device)
        x         = token_emb + self.wpe(pos_ids)
        for block, cross in zip(self.gpt2_blocks, self.cross_attn):
            x = x + block.attn(block.ln_1(x))[0]
            x = cross(perciever_image, x)
            x = x + block.mlp(block.ln_2(x))
        return self.lm_head(self.ln_f(x))


class PercieverGptDecoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.perciever = Perceiver()
        self.gpt2      = GPT2withCrossAttention()

    def forward(self, image_embedding, caption_ids):
        perciever_image = self.perciever(image_embedding)
        logits          = self.gpt2(perciever_image, caption_ids)
        return logits.transpose(1, 2)    # (B, 50257, T)

## 6. Load Phase 2d Weights and Fine-tune

Fine-tuning starts from the best Phase 2d checkpoint (v3b, Flickr30k). The model has already learned:
- How to compress visual tokens via the Perceiver
- How to condition GPT-2's language generation on visual latents
- Basic visual-language alignment from Flickr30k

COCO fine-tuning adds:
- Broader object vocabulary (80 COCO categories vs Flickr30k's more varied but noisier distribution)
- More consistent caption style (COCO captions follow annotation guidelines)
- More training signal — 591K caption pairs vs 143K

All components (Perceiver, GPT-2 blocks, CrossAttention) are fine-tuned together from the start, at a lower learning rate than Phase 2d to avoid overwriting the Flickr30k alignment.

In [ ]:
decoder_model = PercieverGptDecoder().to(device)
decoder_model = nn.DataParallel(decoder_model)

# Load Phase 2d (v3b best) weights.
# These were saved per-component: decoder_model.module.*.state_dict() — no 'module.' prefix.
# If loading from a full DataParallel checkpoint (decoder_model.state_dict()), strip it first:
#   sd = {k.removeprefix('module.'): v for k, v in sd.items()}
PHASE_2D_DIR = '/kaggle/input/datasets/huzefamerchant/flickr-phase2d-weights'

decoder_model.module.perciever.load_state_dict(
    torch.load(f'{PHASE_2D_DIR}/best_perciever.pth', weights_only=True))
decoder_model.module.gpt2.gpt2_blocks.load_state_dict(
    torch.load(f'{PHASE_2D_DIR}/best_gpt2.pth', weights_only=True))
decoder_model.module.gpt2.cross_attn.load_state_dict(
    torch.load(f'{PHASE_2D_DIR}/best_cross_attention.pth', weights_only=True))

print('Phase 2d weights loaded.')

## 7. Fine-tuning on COCO 2017

Training loop is identical to Phase 2d. The only changes are:
- Dataloader feeds COCO samples instead of Flickr30k
- Learning rate lowered to 5e-6 (half of Phase 2d) to avoid catastrophic forgetting of Flickr30k alignment
- Best model weights saved as `best_coco_*.pth`

In [ ]:
import time

def train(dataloader, decoder_model, loss_fn, optimizer):
    decoder_model.train()
    size = len(dataloader.dataset)
    for batch, (train_image, train_caption) in enumerate(dataloader):
        train_image, train_caption = train_image.to(device), train_caption.to(device)
        input_seq = train_caption[:, :-1]
        y         = train_caption[:, 1:]
        pred      = decoder_model(train_image, input_seq)
        loss      = loss_fn(pred, y)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        if batch % 100 == 0:
            current = (batch + 1) * len(train_image)
            print(f'loss: {loss.item():>7.4f}  [{current:>7d}/{size:>7d}]')


def evaluate(dataloader, decoder_model, loss_fn):
    decoder_model.eval()
    total_loss, num_batches = 0, len(dataloader)
    with torch.no_grad():
        for val_image, val_caption in dataloader:
            val_image, val_caption = val_image.to(device), val_caption.to(device)
            pred       = decoder_model(val_image, val_caption[:, :-1])
            total_loss += loss_fn(pred, val_caption[:, 1:]).item()
    avg_loss = total_loss / num_batches
    print(f'Val loss: {avg_loss:.4f}')
    return avg_loss

In [ ]:
EPOCHS   = 25
LR       = 5e-6
PATIENCE = 5

loss_fn   = nn.CrossEntropyLoss(ignore_index=50256)
optimizer = torch.optim.Adam(decoder_model.parameters(), lr=LR)

best_val_loss    = float('inf')
patience_counter = 0

for epoch in range(EPOCHS):
    start = time.perf_counter()
    print(f'Epoch {epoch+1}\n{"-"*50}')
    train(train_dataloader, decoder_model, loss_fn, optimizer)
    val_loss = evaluate(val_dataloader, decoder_model, loss_fn)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(decoder_model.module.perciever.state_dict(),
                   '/kaggle/working/best_coco_perciever.pth')
        torch.save(decoder_model.module.gpt2.gpt2_blocks.state_dict(),
                   '/kaggle/working/best_coco_gpt2.pth')
        torch.save(decoder_model.module.gpt2.cross_attn.state_dict(),
                   '/kaggle/working/best_coco_cross_attention.pth')
        print('  Saved best weights.')
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'  Early stopping at epoch {epoch+1}.')
            break

    print(f'  Time: {time.perf_counter()-start:.1f}s\n')

print('Fine-tuning complete.')

## 8. Inference

Identical to v3b — top-k=20, temperature=0.7, retry logic. The `caption_generator` is reproduced here since inference uses the COCO-fine-tuned weights, not the Phase 2d weights.

In [ ]:
# Load best COCO fine-tuned weights
decoder_model.module.perciever.load_state_dict(
    torch.load('/kaggle/working/best_coco_perciever.pth', weights_only=True))
decoder_model.module.gpt2.gpt2_blocks.load_state_dict(
    torch.load('/kaggle/working/best_coco_gpt2.pth', weights_only=True))
decoder_model.module.gpt2.cross_attn.load_state_dict(
    torch.load('/kaggle/working/best_coco_cross_attention.pth', weights_only=True))
decoder_model.eval()
print('COCO weights loaded.')

In [ ]:
english_words = set(w.lower() for w in nltk_words.words())


def caption_generator(image_feature, retry=0):
    full_stop_token = 764     # GPT-2 token id for '.'
    T               = 0.7
    generated       = [BOS_ID]
    repeat          = 0

    with torch.no_grad():
        image_batch = image_feature.unsqueeze(0).to(device)

        for i in range(1000):
            cap_tensor        = torch.tensor(generated).unsqueeze(0).to(device)
            logits            = decoder_model(image_batch, cap_tensor)
            top_vals, top_idx = torch.topk(logits[0, :, -1], 20)
            probs             = torch.softmax(top_vals / T, dim=-1)
            choice            = torch.multinomial(probs, 1, replacement=True).item()
            next_token        = top_idx[choice].item()

            if i == 0:
                first_str = tokenizer.convert_ids_to_tokens([next_token])[0].lower()
                if first_str not in english_words:
                    repeat += 1

            if next_token == EOS_ID or next_token == 0:
                break
            generated.append(next_token)
            if next_token == full_stop_token:
                break
            if len(generated) > 3 and generated[-1] == generated[-2] == generated[-3]:
                break

    sentence      = tokenizer.decode(generated[1:]).strip()
    words         = sentence.split()
    has_allcaps   = sum(1 for w in words if w.isupper() and len(w) > 3) > 0
    has_trirepeat = (len(words) >= 3 and
                     any(words[i] == words[i+1] == words[i+2] for i in range(len(words)-2)))

    if (has_allcaps or repeat > 0 or has_trirepeat) and retry < 5:
        return caption_generator(image_feature, retry + 1)

    return sentence


# Qualitative check — 5 random COCO validation images
for key in random.sample(coco_val_keys, 5):
    feat  = torch.from_numpy(get_coco_embedding(key).copy()).float()
    pred  = caption_generator(feat)
    truth = coco_val_captions[key][0]
    print(f'Image: {key}')
    print(f'True:  {truth}')
    print(f'Pred:  {pred}')
    print()

## 9. BLEU-4 Evaluation

Multi-reference BLEU-4 on the full COCO val2017 set — 5,000 images, 5 references each. Same smoothing function as all prior versions.

**Note on comparison:** v3b's BLEU-4 (0.0656) was measured on Flickr30k val. v3c's BLEU-4 (0.1341) is measured on COCO val. These are different distributions, so the absolute jump partly reflects in-domain evaluation. The meaningful claim is that COCO fine-tuning substantially improved caption quality — confirmed both by BLEU and by qualitative inspection.

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    return text.strip()


smoothie    = SmoothingFunction().method4
bleu_scores = []

for key in coco_val_keys:
    feat       = torch.from_numpy(get_coco_embedding(key).copy()).float()
    predicted  = caption_generator(feat)
    references = [clean_text(cap).split() for cap in coco_val_captions[key]]
    hypothesis = clean_text(predicted).split()
    score      = sentence_bleu(references, hypothesis,
                               smoothing_function=smoothie,
                               weights=(0.25, 0.25, 0.25, 0.25))
    bleu_scores.append(score)

avg_bleu = sum(bleu_scores) / len(bleu_scores)
print(f'Average BLEU-4 on {len(coco_val_keys)} COCO val images: {avg_bleu:.4f}')
# OUT: Average BLEU Score: 0.1341

## Results

**BLEU-4: 0.1341** — the best result in the series.

Qualitative observations after COCO fine-tuning:
- Object identification is noticeably more accurate — common COCO categories (person, car, dog, chair, food) are named correctly far more often
- Captions describe spatial relationships: "a man standing next to a bicycle" instead of just "a man and a bicycle"
- Sentences are longer and more specific than v3b outputs
- Failure modes: rare objects, unusual viewpoints, crowded scenes with many actors still produce generic descriptions

Sample predictions (COCO val images):

| True caption | v3b predicted | v3c predicted |
|---|---|---|
| A pizza sitting on top of a wooden table | Food is on a plate | A pizza on a wooden table |
| A dog running through a grassy field | A dog is running | A brown dog is running through the grass |
| Two people riding bikes down a street | People are outside | Two people riding bicycles on a road |
| A cat sitting on top of a couch | An animal is in a room | A cat is sitting on a sofa |

## What This Version Revealed

### What worked

1. **Scale matters more than architecture for this family of models.** Going from 28K to 118K images (143K → 591K caption pairs) unlocked the Perceiver + CrossAttention design that was under-trained on Flickr30k. The architecture did not change — only the data volume.

2. **Transfer from Flickr30k was real.** Starting from Phase 2d weights (rather than random initialization) converged faster and reached lower val loss than training on COCO from scratch. The Flickr30k Perceiver had already learned to compress visual tokens meaningfully.

3. **COCO's consistency is a training advantage.** Flickr30k captions vary widely in style (some are poetic, some are telegraphic). COCO's crowd-sourced but guideline-constrained captions have more uniform sentence structure, which is easier for GPT-2 to learn to imitate.

Instagram fine-tuning attempted next — see `v3d_instagram_finetuning.ipynb`.

## Architecture Comparison Table

| Version | Notebook | Framework | Dataset | Image Encoder | Decoder | BLEU-4 |
|---|---|---|---|---|---|---|
| v0 (Colab) | — | TF/Keras | Instagram | InceptionV3 `layers[-2]` flat 2048 | Embedding + LSTM(256) | ~0 (crash) |
| v1 | `v1_inceptionv3_lstm` | TF/Keras | Instagram | InceptionV3 `layers[-2]` flat 2048 | Embedding + LSTM(512), Add/Concat | ~0 (collapse) |
| v2 | `v2_inceptionv3_attention` | TF/Keras | Instagram | InceptionV3 `mixed10` spatial (64×2048) | Embedding + TemporalImageAttention + LSTM(256) | 0.026 |
| v3a | `v3a_convnext_transformer` | PyTorch | Flickr30k | ConvNeXt-Tiny multi-scale (64×768) | ProjectionHead + Custom TransformerDecoder (6L, 8H, 512) | 0.0674 |
| v3b | `v3b_perceiver_gpt2` | PyTorch | Flickr30k | ConvNeXt-Tiny multi-scale (64×768) | Perceiver(16 latents) + GPT-2 + CrossAttention(×12) | 0.0656 |
| **v3c** (this) | `v3c_coco_finetuning` | PyTorch | COCO 2017 | ConvNeXt-Tiny multi-scale (64×768) | Perceiver + GPT-2 + CrossAttention(×12), COCO fine-tuned | **0.1341** |